# Running code

In [1]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity/

!pwd

#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

/home/565/pv3484/aus_substation_electricity
/home/565/pv3484/aus_substation_electricity
processing nsw substations for ['ausgrid'] from None to None
ausgrid
following columns in demand are not in info index:
['MT_HU', 'SI_NO']
removing these columns from demand
number of substations in ausgrid substation info: 134
number of substations in ausgrid substation data: 132
following sites match selection criteria:
               energy_asset          Name  Area  Dwellings  Persons  Residential  Commercial  Industrial  Primary Production  Education  \
ID                                                                                                                                        
BLAKE         AG_BLAKEHURST    Blakehurst     7      10081    28521        0.850       0.005       0.021               0.000      0.022   
PUNCH          AG_PUNCHBOWL     Punchbowl     9      17514    50395        0.826       0.048       0.038               0.000      0.025   
MEADO         AG_MEADOWBANK    M

# Demand Anomaly Animated Map
- combining demand animated map with anomalies

In [4]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
import numpy as np
import os

def animate_holiday_anomalies(demand, info, years, holiday_func, holiday_name,
                              lat_col="latitude", lon_col="longitude",
                              cmap="coolwarm", fps=4,
                              base_dir="/home/565/pv3484/aus_substation_electricity/figures/map_animation/Anomaly"):
    """
    Map and animate 2-year demand anomalies for a holiday.
    Compares anomalies between two years (years[0], years[1]).
    Saves into Anomaly/<HolidayName>/ subfolder.
    """

    demand.index = pd.to_datetime(demand.index)
    hourly = demand.resample("h").mean()

    anomalies_by_year = {}
    for year in years:
        ref_date = holiday_func(year)
        start, end = ref_date - pd.Timedelta(days=30), ref_date + pd.Timedelta(days=30)
        window = hourly.loc[start:end]
        baseline = window.mean()
        holiday_day = hourly.loc[ref_date:ref_date + pd.Timedelta(hours=23)]
        anomalies = holiday_day.subtract(baseline, axis=1)
        anomalies_by_year[year] = anomalies

    # Difference between two years
    diff = anomalies_by_year[years[1]] - anomalies_by_year[years[0]]

    # Merge with metadata
    diff_long = diff.reset_index().melt(id_vars="index", var_name="substation", value_name="anomaly")
    diff_long.rename(columns={"index": "timestamp"}, inplace=True)
    info_reset = info.reset_index().rename(columns={info.index.name or "index": "substation"})
    merged = diff_long.merge(info_reset, on="substation", how="left")

    timestamps = merged["timestamp"].sort_values().unique()
    vmin, vmax = merged["anomaly"].min(), merged["anomaly"].max()

    # Setup figure
    fig, ax = plt.subplots(figsize=(8,6))
    subset0 = merged[merged["timestamp"] == timestamps[0]]
    sc = ax.scatter(subset0[lon_col], subset0[lat_col],
                    c=subset0["anomaly"], cmap=cmap, s=100,
                    vmin=vmin, vmax=vmax)
    plt.colorbar(sc, ax=ax, label="Anomaly Difference (Year2 - Year1)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"{holiday_name} Anomaly Difference: {years[0]} vs {years[1]} at {timestamps[0].strftime('%H:%M')}")

    def update(frame):
        ts = timestamps[frame]
        subset = merged[merged["timestamp"] == ts]
        sc.set_offsets(subset[[lon_col, lat_col]].values)
        sc.set_array(subset["anomaly"].values)
        ax.set_title(f"{holiday_name} Anomaly Difference: {years[0]} vs {years[1]} at {ts.strftime('%H:%M')}")
        return sc,

    ani = animation.FuncAnimation(fig, update, frames=len(timestamps), blit=False)

    # Save to Anomaly/<HolidayName> subfolder
    folder = os.path.join(base_dir, holiday_name.replace(" ", "_"))
    os.makedirs(folder, exist_ok=True)
    filename = f"{holiday_name.replace(' ', '_')}_{years[0]}_{years[1]}_anomaly.gif"
    output_path = os.path.join(folder, filename)
    ani.save(output_path, writer="pillow", fps=fps)
    plt.close(fig)
    return ani

In [5]:
ani = animate_holiday_anomalies(
    demand=demand,
    info=info,
    years=[2010, 2011],
    holiday_func=HOLIDAYS["Christmas Day"],
    holiday_name="Christmas Day",
    cmap="coolwarm",
    fps=6
)

NameError: name 'HOLIDAYS' is not defined